# Metrics: continuous omics

Computes and saves the evaluation metrics for **one** continuous omic, generation method and setting:

| metric | what | file |
|---|---|---|
| Pearson | correlation between ground truth and point prediction, per feature and per observation | `Pearson_Corr/{omic}/{omic}_{feature,sample}_pearson*.csv` |
| RMSE | per feature and per observation | `RMSE/{omic}_{feature,sample}_rmse*.csv` |
| Sharpness | 95% prediction-interval width (PIW) of the samples, per datapoint | `sharpness/{omic}_sharpness*.csv` |
| Calibration | ENCE per feature (10 equal-width bins of predicted std) | `calibration/{omic}_calibration*.csv` |
| Scoring rule | CRPS per feature | `CRPS/{omic}_CRPS*.csv` |

File suffix: `''` latent sampling, `_MCD` MC Dropout, then `_OOD` for the held-out rows.

Everything is set in the next cell; no other cell needs editing. The shared code is in `omics_metrics.py`
(computation) and `omics_config.py` (paths, file names, options). Exploration and plots live in the `exploration_*`
notebooks, which only read the files written here.

In [ ]:
import pandas as pd

import omics_config as cfg
import omics_metrics as om

OMIC = "metabolomics"   # metabolomics | drugresponse | methylation | proteomics | transcriptomics | crisprcas9
METHOD = "latent"       # "latent" (latent sampler) | "mcd" (MC Dropout)
SETTING = "IS"          # "IS" (in-sample) | "OOD" (held-out rows)

cfg.SAVE_OUTPUTS = True        # False: compute and inspect without writing files
cfg.DEBUG_N_FEATURES = None    # e.g. 100 for a quick test on the first 100 features (never saves)

## 1. Data

`om.prepare` returns the ground truth, the point predictions and the sample cube, aligned on the same rows and
features:

* **Ground truth** from `datasets/{omic}.csv`, oriented samples x features. CRISPR-Cas9 and transcriptomics use DepMap
  IDs, which are mapped to Sanger IDs with `Model.csv`; CRISPR-Cas9 gene effects are also scaled so the median
  non-essential gene is 0 and the median essential gene is -1.
* **Predictions and samples** (`METHOD`): latent sampling loads the saved cube; MC Dropout draws 100 samples from
  Normal(mean, var) of the MC Dropout files (seed `cfg.RNG_SEED`).
* **Rows** (`SETTING`): in-sample keeps every sample that has ground truth; OOD loads the predictions of the model
  trained without the held-out rows and keeps only those rows (`Removed_rows/`).
* Features present in both the ground truth and the predictions are kept, and all three are z-scored with the
  per-feature mean/std of the ground truth.

In [ ]:
data, imputed, cube = om.prepare(OMIC, METHOD, SETTING)
print(f"{OMIC} | {METHOD} | {SETTING}")
print(f"ground truth {data.shape}, predictions {imputed.shape}, samples {cube.shape}")

## 2. Pearson correlation

In [ ]:
feature_pearson, sample_pearson = om.pearson(data, imputed)
display(pd.concat([feature_pearson.describe(), sample_pearson.describe()], axis=1))

## 3. RMSE
Positions without ground truth are ignored.

In [ ]:
feature_rmse, sample_rmse, overall_rmse = om.rmse(data, imputed)
print("Overall RMSE:", overall_rmse)
display(pd.concat([feature_rmse.describe(), sample_rmse.describe()], axis=1))

## 4. Sharpness: 95% prediction-interval width (PIW)
PIW = 97.5th - 2.5th percentile of the samples of each datapoint.

In [ ]:
sharpness = om.piw(cube, imputed.index, imputed.columns)
display(sharpness.mean().describe().rename("feature-wise mean PIW"))

## 5. Calibration: ENCE
Per feature, datapoints are binned by their predicted std (10 equal-width bins) and
ENCE = mean over bins of |RMSE_b - RMV_b| / RMV_b.

In [ ]:
calibration = om.feature_wise_ence(data, imputed, cube)["ence"]
display(calibration.describe())

## 6. Scoring rule: CRPS
Empirical CRPS of the samples, averaged over the datapoints of each feature.

In [ ]:
crps = om.feature_wise_crps(data, imputed, cube)["crps"]
display(crps.describe())

## 7. Save
Writes every metric to its usual file and appends a one-line summary to `metrics_summary.csv`.

In [ ]:
results = {
    "feature_pearson": feature_pearson, "sample_pearson": sample_pearson,
    "feature_rmse": feature_rmse, "sample_rmse": sample_rmse,
    "sharpness": sharpness, "calibration": calibration, "crps": crps,
}
om.save_results(results, OMIC, METHOD, SETTING)
summary = om.summarize(results, OMIC, METHOD, SETTING)
om.append_summary(summary)
pd.Series(summary)

## 8. Optional: every omic x method x setting
Runs the same steps as above for all combinations (sections 1-7 are not needed for this). The large omics
(methylation, transcriptomics, crisprcas9) need ~6 GB of memory each.

In [ ]:
RUN_ALL = False

if RUN_ALL:
    del data, imputed, cube
    summaries = []
    for omic in cfg.CONTINUOUS:
        for method in cfg.METHODS:
            for setting in cfg.SETTINGS:
                _, row = om.run(omic, method, setting)
                summaries.append(row)
    display(pd.DataFrame(summaries))